# 📖 Quranic SQL Basics with DuckDB CLI

Simple and clear SQL queries to explore Quran JSON datasets using DuckDB v1.5.5 CLI.

---

## 📂 Datasets

- `quran-metadata-surah-name.json`: Surah info (name, revelation place, verse count).
- `quran-metadata-ayah.json`: Verse info (verse key, word count, text).
- `quran-metadata-juz.json`: 30 Juz divisions.
- `quran-metadata-sajda.json`: Prostration verses.
- `matching-ayah.json`: Verse similarity matches.

## 🛠️ Step 1: Create Views from JSON Files

Run this script once to create clean tables/views from the raw JSON files.

In [ ]:
%%sql

COPY (select revelation_place, avg(verses_count) avg_verse_count, count(*) surah_count From surahs group by revelation_place order by surah_count desc)
 TO 'data/summary_by_revelation_place.parquet' (FORMAT parquet);

-- Create simple views for easy querying
CREATE OR REPLACE VIEW surahs AS 
SELECT key::int AS id, value->>'name_simple' AS name, value->>'revelation_place' AS revelation_place, 
    (value->>'verses_count')::int AS verses_count 
FROM read_json_objects('data/quran-metadata-surah-name.json'), json_each(json);

CREATE OR REPLACE VIEW ayahs AS 
SELECT (value->>'surah_number')::int AS surah_id, (value->>'words_count')::int AS words_count 
FROM read_json_objects('data/quran-metadata-ayah.json'), json_each(json);

CREATE OR REPLACE VIEW juz AS 
SELECT key::int AS juz_number, (value->>'verses_count')::int AS verses_count, value->>'first_verse_key' AS start_verse, value->>'last_verse_key' AS end_verse 
FROM read_json_objects('data/quran-metadata-juz.json'), json_each(json);

CREATE OR REPLACE VIEW sajdah AS 
SELECT (value->>'sajdah_number')::int AS sajdah_number, value->>'verse_key' AS verse_key, value->>'sajdah_type' AS sajdah_type, split_part(value->>'verse_key', ':', 1)::int AS surah_id 
FROM read_json_objects('data/quran-metadata-sajda.json'), json_each(json);

CREATE OR REPLACE VIEW matches AS 
SELECT m_raw.key AS source_verse, m.value->>'matched_ayah_key' AS matched_verse, (m.value->>'score')::int AS score 
FROM read_json_objects('data/matching-ayah.json'), json_each(json) AS m_raw, json_each(m_raw.value) AS m;

### Question 1: Surahs and Verses by Revelation Place

Count total surahs and total verses for Makkah vs Madinah.

**SQL Concepts:** `GROUP BY`, `COUNT`, `SUM`.

In [ ]:
%%sql
-- Q1: Total surahs and verses by revelation place
SELECT revelation_place, COUNT(*) AS total_surahs, SUM(verses_count) AS total_verses
FROM surahs
GROUP BY revelation_place;

### Question 2: Surahs with Prostration Verses (Sajdah)

List all 15 Sajdah verses with their Surah names.

**SQL Concepts:** `JOIN`, `ORDER BY`.

In [ ]:
%%sql
-- Q2: List sajdah verses with surah names
SELECT s.sajdah_number, s.verse_key, s.sajdah_type, su.name AS surah_name
FROM sajdah s
JOIN surahs su ON s.surah_id = su.id
ORDER BY s.sajdah_number;

### Question 3: Top 5 Longest Juz by Verse Count

Find the 5 Juz with the most verses.

**SQL Concepts:** `ORDER BY`, `LIMIT`.

In [ ]:
%%sql
-- Q3: Top 5 Juz with the most verses
SELECT juz_number, verses_count, start_verse, end_verse
FROM juz
ORDER BY verses_count DESC
LIMIT 5;

### Question 4: Top 5 Surahs by Total Word Count

Find the 5 largest Surahs by total word count.

**SQL Concepts:** `JOIN`, `GROUP BY`, `SUM`.

In [ ]:
%%sql
-- Q4: Top 5 surahs by total word count
SELECT su.name AS surah_name, su.revelation_place, SUM(a.words_count) AS total_words
FROM surahs su
    JOIN ayahs a ON su.id = a.surah_id
GROUP BY su.name, su.revelation_place
ORDER BY total_words DESC
LIMIT 5;

### Question 5: Find Verse Matches (Mutashabihat)

Find verses with high similarity score (score >= 80).

**SQL Concepts:** `WHERE`, `ORDER BY`, `LIMIT`.

In [ ]:
%%sql
-- Q5: Top matching verses by similarity score
SELECT source_verse, matched_verse, score
FROM matches
WHERE score >= 80
ORDER BY score DESC
LIMIT 10;